# 04 — Training: EfficientNet-B3

Entrenamiento de **EfficientNet-B3** preentrenado en ImageNet para clasificación morfológica de galaxias (6 clases).

**Configuración de hardware:** T4 ×2 (Kaggle) con `DataParallel` + AMP (float16).

**Checkpointing:** se guarda `latest.pth` en cada epoch y `best.pth` cuando mejora el val F1-macro.  
Para reanudar el entrenamiento desde el último punto, ejecutar el notebook sin borrar `latest.pth`.

| Hiperparámetro | Valor |
|---|---|
| Batch size | 128 (64/GPU) |
| LR backbone | 1e-4 |
| LR head | 1e-3 |
| Optimizer | AdamW (wd=1e-4) |
| Scheduler | CosineAnnealingLR |
| Epochs | 30 |
| AMP | ✅ float16 |


## Sección 1 — Imports

In [ ]:
import gc
import os
import sys
import time
import pathlib
import warnings
from datetime import datetime

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import torch
import torch.nn as nn
import torch.optim as optim
from torch.cuda.amp import autocast, GradScaler
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms
from torchvision.models import efficientnet_b3, EfficientNet_B3_Weights
from sklearn.metrics import f1_score, classification_report, confusion_matrix
from PIL import Image
from tqdm.auto import tqdm

warnings.filterwarnings('ignore')

print(f'PyTorch  : {torch.__version__}')
print(f'CUDA     : {torch.cuda.is_available()}')
print(f'GPUs     : {torch.cuda.device_count()}')
for i in range(torch.cuda.device_count()):
    print(f'  GPU {i} : {torch.cuda.get_device_name(i)}')
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device   : {device}')


## Sección 2 — Configuración

In [ ]:
IS_KAGGLE = pathlib.Path('/kaggle/input').exists()

if IS_KAGGLE:
    _IMG_ROOT  = pathlib.Path('/kaggle/input/datasets/jaimetrickz/galaxy-zoo-2-images')
    IMAGES_DIR = _IMG_ROOT / 'images_gz2' / 'images'
    SPLITS_DIR = pathlib.Path('/kaggle/input/datasets/jeancdevx/galaxy-morph-splits')
    CKPT_DIR   = pathlib.Path('/kaggle/working/checkpoints/efficientnet_b3')
    LOG_DIR    = pathlib.Path('/kaggle/working/logs')
else:
    _LOCAL     = pathlib.Path('../data')
    IMAGES_DIR = _LOCAL / 'images_gz2' / 'images'
    SPLITS_DIR = _LOCAL / 'splits'
    CKPT_DIR   = pathlib.Path('../models/checkpoints/efficientnet_b3')
    LOG_DIR    = pathlib.Path('../logs')

CKPT_DIR.mkdir(parents=True, exist_ok=True)
LOG_DIR.mkdir(parents=True, exist_ok=True)

# Reanudar desde una sesión anterior (kernel muerto)
# Si el kernel murió, /kaggle/working/ se pierde con los checkpoints.
# Workflow para recuperarlos:
#   1. Ve a la sesión anterior de Kaggle → Output → descarga latest.pth / best.pth
#   2. Súbelos a un Kaggle Dataset (ej: jeancdevx/galaxy-morph-ckpt-efficientnet-b3)
#   3. Añade ese dataset como Input en este notebook
#   4. Pon el slug del dataset abajo (la parte después de /kaggle/input/)
RESUME_DATASET = None  # ← ej: 'jeancdevx/galaxy-morph-ckpt-efficientnet-b3'

# Model
MODEL_NAME   = 'efficientnet_b3'
NUM_CLASSES  = 6
CLASS_ORDER  = ['Elliptical', 'Lenticular', 'Spiral', 'Barred_Spiral', 'Edge_on', 'Irregular']
CLASS_TO_IDX = {c: i for i, c in enumerate(CLASS_ORDER)}
IDX_TO_CLASS = {i: c for c, i in CLASS_TO_IDX.items()}

# Training
EPOCHS       = 30
BATCH_SIZE   = 128    # 64/GPU on T4×2 con AMP
NUM_WORKERS  = 2 if IS_KAGGLE else 0
LR_BACKBONE  = 1e-4   # conservador: backbone ya preentrenado
LR_HEAD      = 1e-3   # agresivo: head inicializado al azar
WEIGHT_DECAY = 1e-4
USE_AMP      = device.type == 'cuda'

# Image
IMAGE_SIZE    = 224
CROP_SIZE     = 320
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]
RANDOM_SEED   = 42

torch.manual_seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

print(f'IS_KAGGLE      : {IS_KAGGLE}')
print(f'IMAGES_DIR     : {IMAGES_DIR}')
print(f'SPLITS_DIR     : {SPLITS_DIR}')
print(f'CKPT_DIR       : {CKPT_DIR}')
print(f'RESUME_DATASET : {RESUME_DATASET or "(ninguno — entrenamiento nuevo o reanudar desde working)"}')
print(f'MODEL          : {MODEL_NAME}')
print(f'EPOCHS         : {EPOCHS}')
print(f'BATCH_SIZE     : {BATCH_SIZE}')
print(f'NUM_WORKERS    : {NUM_WORKERS}')
print(f'LR backbone    : {LR_BACKBONE}')
print(f'LR head        : {LR_HEAD}')
print(f'USE_AMP        : {USE_AMP}')


## Sección 3 — Pipeline de datos

In [ ]:
# Transforms
train_transforms = transforms.Compose([
    transforms.CenterCrop(CROP_SIZE),
    transforms.Resize(IMAGE_SIZE),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomVerticalFlip(p=0.5),
    transforms.RandomRotation(degrees=180),
    transforms.ColorJitter(brightness=0.15, contrast=0.15),
    transforms.ToTensor(),
    transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
])

eval_transforms = transforms.Compose([
    transforms.CenterCrop(CROP_SIZE),
    transforms.Resize(IMAGE_SIZE),
    transforms.ToTensor(),
    transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
])


# Dataset
class GalaxyDataset(Dataset):
    def __init__(self, csv_path, images_dir, transform, class_to_idx):
        df = pd.read_csv(
            csv_path,
            usecols=['img_filename', 'morph_label'],
            dtype={'img_filename': 'str', 'morph_label': 'str'},
        )
        self.filenames = df['img_filename'].to_numpy()
        self.labels    = np.array(
            [class_to_idx[lbl] for lbl in df['morph_label']], dtype=np.int64
        )
        del df
        gc.collect()

        self.images_dir = pathlib.Path(images_dir)
        self.transform  = transform

    def __len__(self):
        return len(self.filenames)

    def __getitem__(self, idx):
        image = Image.open(self.images_dir / self.filenames[idx]).convert('RGB')
        if self.transform:
            image = self.transform(image)
        return image, int(self.labels[idx])


# DataLoaders
g = torch.Generator().manual_seed(RANDOM_SEED)

train_loader = DataLoader(
    GalaxyDataset(SPLITS_DIR / 'train.csv', IMAGES_DIR, train_transforms, CLASS_TO_IDX),
    batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS,
    pin_memory=(device.type == 'cuda'), generator=g,
)
val_loader = DataLoader(
    GalaxyDataset(SPLITS_DIR / 'val.csv', IMAGES_DIR, eval_transforms, CLASS_TO_IDX),
    batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS,
    pin_memory=(device.type == 'cuda'),
)

print(f'Train batches : {len(train_loader):,}  ({len(train_loader.dataset):,} imgs)')
print(f'Val   batches : {len(val_loader):,}  ({len(val_loader.dataset):,} imgs)')


In [ ]:
# Class weights para CrossEntropyLoss
_df_w = pd.read_csv(
    SPLITS_DIR / 'train.csv',
    usecols=['morph_label'],
    dtype={'morph_label': 'str'},
)
label_counts = np.bincount(
    _df_w['morph_label'].map(CLASS_TO_IDX).values,
    minlength=NUM_CLASSES,
)
weights      = len(_df_w) / (NUM_CLASSES * label_counts)
class_weights = torch.tensor(weights, dtype=torch.float32).to(device)

print('Class weights:')
for cls, w, n in zip(CLASS_ORDER, weights, label_counts):
    print(f'  {cls:<15}  n={n:>6,}   w={w:.4f}')

del _df_w
gc.collect()
print('RAM liberada ✓')


## Sección 4 — Modelo

In [ ]:
# Cargar EfficientNet-B3 con pesos preentrenados en ImageNet
base_model = efficientnet_b3(weights=EfficientNet_B3_Weights.IMAGENET1K_V1)

# Reemplazar la cabeza de clasificación (1536 → NUM_CLASSES)
in_features = base_model.classifier[1].in_features
base_model.classifier[1] = nn.Linear(in_features, NUM_CLASSES)
print(f'Classifier head: Linear({in_features}, {NUM_CLASSES})')

# Parámetros por grupo ANTES de DataParallel
backbone_params = [p for n, p in base_model.named_parameters()
                   if not n.startswith('classifier')]
head_params     = list(base_model.classifier.parameters())
print(f'Backbone params : {sum(p.numel() for p in backbone_params):,}')
print(f'Head params     : {sum(p.numel() for p in head_params):,}')

# DataParallel si hay 2 GPUs (T4×2)
n_gpus = torch.cuda.device_count()
if n_gpus > 1:
    model = nn.DataParallel(base_model)
    print(f'DataParallel across {n_gpus} GPUs')
else:
    model = base_model

model = model.to(device)
total_params = sum(p.numel() for p in model.parameters())
print(f'Total params    : {total_params:,}')

## Sección 5 — Infraestructura de entrenamiento

In [ ]:
# Loss con pesos de clase para compensar el desbalance residual
criterion = nn.CrossEntropyLoss(weight=class_weights)

# Optimizer con LR diferencial: backbone conservador, cabeza agresiva
optimizer = optim.AdamW(
    [
        {'params': backbone_params, 'lr': LR_BACKBONE},
        {'params': head_params,     'lr': LR_HEAD},
    ],
    weight_decay=WEIGHT_DECAY,
)

# Scheduler: cosine decay hasta eta_min al final del entrenamiento
scheduler = optim.lr_scheduler.CosineAnnealingLR(
    optimizer, T_max=EPOCHS, eta_min=1e-6
)

# AMP GradScaler (solo activo si hay GPU)
scaler = GradScaler(enabled=USE_AMP)

print('Loss     : CrossEntropyLoss (weighted)')
print('Optim    : AdamW')
print('Schedule : CosineAnnealingLR')
print(f'AMP      : {USE_AMP}')

In [ ]:
# Funciones de checkpoint
def save_checkpoint(state: dict, ckpt_dir: pathlib.Path, is_best: bool = False):
    """
    Guarda el estado completo del entrenamiento.
    - latest.pth     : sobreescrito en cada epoch (para reanudar)
    - best.pth       : solo cuando mejora el val F1
    - epoch_XXX.pth  : hito cada 5 epochs (archivo permanente)
    """
    ckpt_dir.mkdir(parents=True, exist_ok=True)
    torch.save(state, ckpt_dir / 'latest.pth')
    saved = ['latest.pth']
    if is_best:
        torch.save(state, ckpt_dir / 'best.pth')
        saved.append('best.pth')
    if (state['epoch'] + 1) % 5 == 0:
        name = f'epoch_{state["epoch"]+1:03d}.pth'
        torch.save(state, ckpt_dir / name)
        saved.append(name)
    print(f'  [ckpt] saved: {", ".join(saved)}')


def load_checkpoint(
    ckpt_path: pathlib.Path,
    model: nn.Module,
    optimizer: optim.Optimizer,
    scheduler,
    scaler: GradScaler,
):
    """
    Restaura modelo, optimizer, scheduler, scaler e historial completo.
    Devuelve (last_epoch, best_val_f1, history).
    """
    ckpt      = torch.load(ckpt_path, map_location='cpu')
    raw_model = model.module if isinstance(model, nn.DataParallel) else model
    raw_model.load_state_dict(ckpt['model_state_dict'])
    optimizer.load_state_dict(ckpt['optimizer_state_dict'])
    scheduler.load_state_dict(ckpt['scheduler_state_dict'])
    if 'scaler_state_dict' in ckpt:
        scaler.load_state_dict(ckpt['scaler_state_dict'])
    return ckpt['epoch'], ckpt['best_val_f1'], ckpt['history']


print('save_checkpoint / load_checkpoint definidas ✓')

## Sección 6 — Reanudar desde checkpoint

Si existe `latest.pth`, se restaura todo el estado y el entrenamiento continúa desde el siguiente epoch.  
Si no existe, comienza desde cero.

In [ ]:
LATEST_CKPT = CKPT_DIR / 'latest.pth'

# Si no hay checkpoint en /kaggle/working/ (ej: kernel murió en sesión anterior),
# busca en el dataset de Kaggle que el usuario subió manualmente.
if not LATEST_CKPT.exists() and IS_KAGGLE and RESUME_DATASET:
    candidate = pathlib.Path(f'/kaggle/input/{RESUME_DATASET}/latest.pth')
    if candidate.exists():
        LATEST_CKPT = candidate
        print(f'[resume] Checkpoint hallado en dataset de entrada:\n  {LATEST_CKPT}')
    else:
        print(f'[resume] RESUME_DATASET configurado pero no se encontró latest.pth en:\n  {candidate}')
        print('         Verifica el slug del dataset y que hayas añadido el dataset como Input.')

start_epoch = 0
best_val_f1 = 0.0
history     = []          # lista de dicts, uno por epoch completado

if LATEST_CKPT.exists():
    print(f'Checkpoint encontrado: {LATEST_CKPT}')
    last_epoch, best_val_f1, history = load_checkpoint(
        LATEST_CKPT, model, optimizer, scheduler, scaler
    )
    start_epoch = last_epoch + 1
    print(f'Reanudando desde epoch {start_epoch + 1}/{EPOCHS}')
    print(f'Mejor val F1 hasta ahora : {best_val_f1:.4f}')
    print(f'Epochs en historial      : {len(history)}')
else:
    print('Sin checkpoint — entrenamiento desde cero')

epochs_remaining = EPOCHS - start_epoch
print(f'Epochs por entrenar      : {epochs_remaining}')


## Sección 7 — Funciones de entrenamiento y validación

In [ ]:
def train_one_epoch(model, loader, optimizer, criterion, scaler, device, use_amp):
    model.train()
    running_loss = 0.0
    all_preds, all_labels = [], []

    pbar = tqdm(loader, desc='  Train', unit='batch', leave=False,
                dynamic_ncols=True)
    for imgs, labels in pbar:
        imgs   = imgs.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)

        optimizer.zero_grad()
        with autocast(enabled=use_amp):
            logits = model(imgs)
            loss   = criterion(logits, labels)

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        running_loss += loss.item() * imgs.size(0)
        all_preds.extend(logits.argmax(dim=1).detach().cpu().tolist())
        all_labels.extend(labels.cpu().tolist())

        pbar.set_postfix(loss=f'{loss.item():.4f}')

    avg_loss = running_loss / len(loader.dataset)
    f1       = f1_score(all_labels, all_preds, average='macro', zero_division=0)
    return avg_loss, f1


@torch.no_grad()
def validate(model, loader, criterion, device, use_amp):
    model.eval()
    running_loss = 0.0
    all_preds, all_labels = [], []

    pbar = tqdm(loader, desc='  Val  ', unit='batch', leave=False,
                dynamic_ncols=True)
    for imgs, labels in pbar:
        imgs   = imgs.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)
        with autocast(enabled=use_amp):
            logits = model(imgs)
            loss   = criterion(logits, labels)
        running_loss += loss.item() * imgs.size(0)
        all_preds.extend(logits.argmax(dim=1).cpu().tolist())
        all_labels.extend(labels.cpu().tolist())

        pbar.set_postfix(loss=f'{loss.item():.4f}')

    avg_loss = running_loss / len(loader.dataset)
    f1       = f1_score(all_labels, all_preds, average='macro', zero_division=0)
    return avg_loss, f1, all_preds, all_labels


print('train_one_epoch / validate definidas ✓')


## Sección 8 — Loop de entrenamiento

In [ ]:
LOG_CSV = LOG_DIR / f'{MODEL_NAME}_log.csv'

print(f'Iniciando entrenamiento: epochs {start_epoch+1} → {EPOCHS}')
print('=' * 72)

for epoch in range(start_epoch, EPOCHS):
    # Liberar memoria antes de cada epoch: evita acumulación entre epochs
    gc.collect()
    if device.type == 'cuda':
        torch.cuda.empty_cache()

    t0 = time.time()
    ts = datetime.now().strftime('%H:%M:%S')

    print(f'\n[{ts}] ── Epoch {epoch+1:02d}/{EPOCHS} ───────────────────────────────')

    train_loss, train_f1 = train_one_epoch(
        model, train_loader, optimizer, criterion, scaler, device, USE_AMP
    )
    val_loss, val_f1, val_preds, val_labels = validate(
        model, val_loader, criterion, device, USE_AMP
    )
    scheduler.step()

    elapsed = time.time() - t0
    is_best  = val_f1 > best_val_f1
    if is_best:
        best_val_f1 = val_f1

    # Registro del epoch
    row = {
        'epoch':       epoch + 1,
        'train_loss':  round(train_loss, 6),
        'train_f1':    round(train_f1,   6),
        'val_loss':    round(val_loss,   6),
        'val_f1':      round(val_f1,     6),
        'lr_backbone': round(optimizer.param_groups[0]['lr'], 8),
        'lr_head':     round(optimizer.param_groups[1]['lr'], 8),
        'elapsed_s':   round(elapsed, 1),
        'is_best':     is_best,
    }
    history.append(row)

    # Checkpoint (cada epoch)
    state = {
        'epoch':                epoch,
        'model_state_dict':     (
            model.module if isinstance(model, nn.DataParallel) else model
        ).state_dict(),
        'optimizer_state_dict': optimizer.state_dict(),
        'scheduler_state_dict': scheduler.state_dict(),
        'scaler_state_dict':    scaler.state_dict(),
        'best_val_f1':          best_val_f1,
        'history':              history,
    }
    save_checkpoint(state, CKPT_DIR, is_best=is_best)

    # Persistir CSV de log (sobrevive a caídas del kernel)
    pd.DataFrame(history).to_csv(LOG_CSV, index=False)

    best_tag = '  ← BEST' if is_best else ''
    print(
        f'  train  loss={train_loss:.4f}  F1={train_f1:.4f}\n'
        f'  val    loss={val_loss:.4f}  F1={val_f1:.4f}{best_tag}\n'
        f'  time   {elapsed:.0f}s   LR_bb={optimizer.param_groups[0]["lr"]:.2e}',
        flush=True,
    )

print('\n' + '=' * 72)
print(f'Entrenamiento completo. Mejor val F1 = {best_val_f1:.4f}')


## Sección 9 — Curvas de entrenamiento

In [ ]:
hist_df = pd.DataFrame(history)

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle(f'{MODEL_NAME} — Training History', fontsize=13, fontweight='bold')

# Loss
axes[0].plot(hist_df['epoch'], hist_df['train_loss'], label='Train')
axes[0].plot(hist_df['epoch'], hist_df['val_loss'],   label='Val')
axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('Loss')
axes[0].set_title('Cross-Entropy Loss'); axes[0].legend(); axes[0].grid(alpha=0.3)

# F1
best_row = hist_df.loc[hist_df['val_f1'].idxmax()]
axes[1].plot(hist_df['epoch'], hist_df['train_f1'], label='Train')
axes[1].plot(hist_df['epoch'], hist_df['val_f1'],   label='Val')
axes[1].axvline(best_row['epoch'], color='red', linestyle='--', alpha=0.5,
                label=f'Best={best_row["val_f1"]:.4f} (ep{int(best_row["epoch"])})')
axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('F1 macro')
axes[1].set_title('Macro F1 Score'); axes[1].legend(); axes[1].grid(alpha=0.3)

# LR
axes[2].semilogy(hist_df['epoch'], hist_df['lr_backbone'], label='Backbone')
axes[2].semilogy(hist_df['epoch'], hist_df['lr_head'],     label='Head')
axes[2].set_xlabel('Epoch'); axes[2].set_ylabel('Learning Rate (log)')
axes[2].set_title('Learning Rate Schedule'); axes[2].legend(); axes[2].grid(alpha=0.3)

plt.tight_layout()
plt.savefig(LOG_DIR / f'{MODEL_NAME}_training_curves.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'Figura guardada en {LOG_DIR}/{MODEL_NAME}_training_curves.png')

## Sección 10 — Evaluación final (mejor modelo sobre val)

In [ ]:
# Cargar el mejor checkpoint para la evaluación final
best_ckpt = CKPT_DIR / 'best.pth'
if best_ckpt.exists():
    ckpt      = torch.load(best_ckpt, map_location='cpu')
    raw_model = model.module if isinstance(model, nn.DataParallel) else model
    raw_model.load_state_dict(ckpt['model_state_dict'])
    print(f'Mejor modelo: epoch {ckpt["epoch"]+1}  val F1={ckpt["best_val_f1"]:.4f}')
else:
    print('best.pth no encontrado — usando el estado actual del modelo')

# test_loader se crea aquí (no en Sección 3) para evitar tener workers extra
# vivos durante todo el entrenamiento.
test_loader = DataLoader(
    GalaxyDataset(SPLITS_DIR / 'test.csv', IMAGES_DIR, eval_transforms, CLASS_TO_IDX),
    batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS,
    pin_memory=(device.type == 'cuda'),
)
print(f'Test  batches : {len(test_loader):,}  ({len(test_loader.dataset):,} imgs)')

_, _, val_preds, val_labels = validate(model, val_loader, criterion, device, USE_AMP)

print('\nClassification Report — Val set:')
print(classification_report(val_labels, val_preds, target_names=CLASS_ORDER, zero_division=0))


In [ ]:
# Confusion matrix normalizada
cm      = confusion_matrix(val_labels, val_preds)
cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
fig.suptitle(f'{MODEL_NAME} — Confusion Matrix (val set)', fontsize=12, fontweight='bold')

for ax, data, title, fmt in [
    (axes[0], cm,      'Counts',     'd'),
    (axes[1], cm_norm, 'Normalized', '.2f'),
]:
    sns.heatmap(
        data, annot=True, fmt=fmt, cmap='Blues',
        xticklabels=CLASS_ORDER, yticklabels=CLASS_ORDER,
        ax=ax, vmin=0, vmax=(1 if fmt == '.2f' else None),
    )
    ax.set_title(title)
    ax.set_xlabel('Predicted')
    ax.set_ylabel('True')
    ax.set_xticklabels(CLASS_ORDER, rotation=30, ha='right')
    ax.set_yticklabels(CLASS_ORDER, rotation=0)

plt.tight_layout()
plt.savefig(LOG_DIR / f'{MODEL_NAME}_confusion_matrix.png', dpi=150, bbox_inches='tight')
plt.show()

## Sección 11 — Resumen

**Artefactos generados:**
- `checkpoints/efficientnet_b3/latest.pth` — último checkpoint (reanudar desde aquí)
- `checkpoints/efficientnet_b3/best.pth` — mejor checkpoint por val F1
- `checkpoints/efficientnet_b3/epoch_XXX.pth` — hitos cada 5 epochs
- `logs/efficientnet_b3_log.csv` — historial epoch por epoch
- `logs/efficientnet_b3_training_curves.png`
- `logs/efficientnet_b3_confusion_matrix.png`

**Para reanudar el entrenamiento:**  
Ejecutar el notebook de nuevo sin borrar `latest.pth`. El entrenamiento continuará desde el último epoch completado.

**Para ampliar epochs:**  
Cambiar `EPOCHS = 50` (o el valor deseado) en la Sección 2, y ejecutar el notebook — el checkpoint restaura el scheduler correctamente.

**Siguiente paso → `05_train_resnet50.ipynb`**